In [10]:
# Task 3: Data Cleaning & Preprocessing (Titanic Dataset)

## Objective
The goal of this notebook is to take the raw Titanic dataset and transform it into a clean, analysis-ready format through a documented,
step-by-step cleaning workflow.

In [12]:
import numpy as np
import pandas as pd

# Configure pandas display settings for clean table rendering
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Load the raw dataset
df_raw = pd.read_csv("Titanic-Dataset.csv")
df = df_raw.copy()

print(f"Dataset loaded successfully with shape: {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

Dataset loaded successfully with shape: 891 rows and 12 columns.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
## Step 1: Initial Data Quality Report

Before making any modifications, we perform an automated audit across all 12 columns to inspect;
- Total missing values and missing percentage
- Unique values count
- Detected data type (dtype)
- Value range anomalies (minimum and maximum observed values)
- Total duplicate rows

In [13]:
# Check total duplicate rows across all columns
initial_duplicates = df.duplicated().sum()

# Construct data quality summary table
quality_report = pd.DataFrame(
    {
        "Data Type": df.dtypes,
        "Null Count": df.isnull().sum(),
        "Null Pct (%)": (df.isnull().sum() / len(df) * 100).round(2),
        "Unique Values": df.nunique(),
        "Min Value": [
            df[col].dropna().min() if df[col].dropna().size > 0 else np.nan
            for col in df.columns
        ],
        "Max Value": [
            df[col].dropna().max() if df[col].dropna().size > 0 else np.nan
            for col in df.columns
        ],
    }
)

print(f"Total duplicate rows detected: {initial_duplicates}")
quality_report

Total duplicate rows detected: 0


,Data Type,Null Count,Null Pct (%),Unique Values,Min Value,Max Value
PassengerId,int64,0,0.00,891,1,891
Survived,int64,0,0.00,2,0,1
Pclass,int64,0,0.00,3,1,3
Name,str,0,0.00,891,"Abbing, Mr. Anthony","van Melkebeke, Mr. Philemon"
Sex,str,0,0.00,2,female,male
Age,float64,177,19.87,88,0.42,80.0
SibSp,int64,0,0.00,7,0,8
Parch,int64,0,0.00,7,0,6
Ticket,str,0,0.00,681,110152,WE/P 5735
Fare,float64,0,0.00,248,0.0,512.3292


In [ ]:
### Key Findings from Data Quality Audit:
1. Missing Data:
   - Cabin: Missing 687 values (~77.1%). Dropping rows would discard most of the dataset.
    Because the cabin deck letter carries historical survival significance,
    we will extract the deck initial and impute missing entries with Unknown.
   - Age: Missing 177 values (~19.87%). Because age is slightly right-skewed and correlates with passenger title/class,
     imputing by median grouped by passenger title preserves demographic fidelity without outlier distortion.
   - `Embarked: Missing 2 entries (~0.22%). Mode imputation is ideal given the tiny missing share.
2. Duplicate Records: 0 exact duplicate rows found across all fields.
3. Data Types:
   - "PassengerId" is an integer identifier; it should be stored as an object/string.
   - "Pclass" and "Survived" are categorical flags represented as integers; casting them appropriately ensures model clarity.
   - "Fare" has a minimum value of "0.0" (crew/complimentary tickets) and a maximum of "512.33" (first-class luxury suites).

In [ ]:
## Step 2: Missing Data Handling

### Strategy Justifications:
- Embarked (Categorical, 2 missing): We use mode imputaion. 'S' (Southampton) represents over 72% of the boarding population,
  so imputing 'S' has negligible impact on the joint distribution.
- Age (Continuous, 177 missing): Overall median age is 28.0, but titles like "Master" (children) versus "Mr" / "Mrs" / "Doctor" 
  have distinct age distribution. We extract passenger title ('Mr', 'Mrs', 'Miss', 'Master', etc.)
  and impute the median age of that specific title group.
- Cabin (Alphanumeric, 687 missing): Dropping rows causes massive data loss. Instead, we impute missing records with 'Unknown'
  and extract the first letter (Deck code: A, B, C, D, E, F, G, T, Unknown) to prserve spatial information.

In [14]:
# 1. Impute Embarked with the mode
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)
print(f"Imputed Embarked missing values with mode: '{embarked_mode}'")

# 2. Extract Title from Name to perform group-median imputation for Age
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()

# Group infrequent titles into common brackets or 'Other'
title_mapping = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Royalty",
    "the Countess": "Royalty",
    "Countess": "Royalty",
    "Capt": "Officer",
    "Col": "Officer",
    "Major": "Officer",
    "Dr": "Officer",
    "Rev": "Officer",
    "Sir": "Royalty",
    "Jonkheer": "Royalty",
    "Don": "Royalty",
    "Dona": "Royalty",
}
df["Title"] = df["Title"].replace(title_mapping)

# Calculate title-based median ages and fill missing Age
title_medians = df.groupby("Title")["Age"].median()
print("\nMedian age per passenger title group:")
print(title_medians)

df["Age"] = df.groupby("Title")["Age"].transform(lambda x: x.fillna(x.median()))
# Fallback in case any title has all nulls
df["Age"] = df["Age"].fillna(df["Age"].median())

# 3. Impute Cabin with 'Unknown' and extract Deck initial
df["Cabin"] = df["Cabin"].fillna("Unknown")
df["Deck"] = df["Cabin"].apply(lambda x: x[0] if x != "Unknown" else "Unknown")

print("\nNull counts after imputation:")
print(df[["Age", "Cabin", "Embarked"]].isnull().sum())

Imputed Embarked missing values with mode: 'S'

Median age per passenger title group:
Title
Master      3.5
Miss       21.0
Mr         30.0
Mrs        35.0
Officer    50.0
Royalty    40.0
Name: Age, dtype: float64

Null counts after imputation:
Age         0
Cabin       0
Embarked    0
dtype: int64


In [ ]:
## Step 3: Duplicate Row Detection and Removal

We check for duplicate records both globally across all columns and on passenger identity fields (Name, Ticket, Fare, Sex).
Any verified duplicates are dropped, retaining the first occurrence.

In [15]:
# Check total full-row duplicates
exact_dupes = df.duplicated().sum()
print(f"Exact full-row duplicates detected: {exact_dupes}")

# Check identity-based duplicates (excluding auto-increment PassengerId)
id_dupes = df.duplicated(subset=["Name", "Sex", "Age", "Ticket"]).sum()
print(f"Identity duplicates (Name, Sex, Age, Ticket): {id_dupes}")

# Execute duplicate drop if any were present
before_count = len(df)
df = df.drop_duplicates()
removed_count = before_count - len(df)
print(f"Rows removed during duplicate purging: {removed_count}")

Exact full-row duplicates detected: 0
Identity duplicates (Name, Sex, Age, Ticket): 0
Rows removed during duplicate purging: 0


In [ ]:
## Step 4: Text Standardisation and Categorical Normalisation

Inconsistent casing, stray whitespace, or irregular abbreviations can distort categorical grouping:
- Sex: Strip witespace, convert to lowercase, and standardize to title case ('male' / 'female' $\rightarrow$ 'Male', 'Female').
- Embarked: Convert port codes to standrd upper-case single-character identifiers ('C', 'Q', 'S').
- Name / Ticket: Strip leading/trailing whitespaces.

In [16]:
# Standardize Sex column
df["Sex"] = df["Sex"].astype(str).str.strip().str.lower()
sex_standard_map = {
    "male": "Male",
    "m": "Male",
    "female": "Female",
    "f": "Female",
}
df["Sex"] = df["Sex"].map(sex_standard_map).fillna(df["Sex"].str.capitalize())

# Standardize Embarked column
df["Embarked"] = df["Embarked"].astype(str).str.strip().str.upper()

# Clean Ticket and Name whitespace
df["Ticket"] = df["Ticket"].astype(str).str.strip()
df["Name"] = df["Name"].astype(str).str.strip()

print("Standardised unique values for 'Sex':", df["Sex"].unique())
print("Standardised unique values for 'Embarked':", df["Embarked"].unique())

Standardised unique values for 'Sex': <StringArray>
['Male', 'Female']
Length: 2, dtype: str
Standardised unique values for 'Embarked': <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str


In [ ]:
## Step 5: Outlier Detection and Treatment (IQR Method)

We inspect the numeric variables (Age and Fare) for outliers using the Interquartile Range (IQR) rule:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

### Treatment Decisions:
- Age (Retain): The upper bound is ~65 years. The oldest passengers aboard were aged 70–80 (e.g., Mr. Algernon Barkworth, aged 80).
  These are historically accurate records and biologically valid human ages,
  not data entry errors. Removing or capping them woud distort age-related mortality analysis.
- Fare (Cap / Winsorize): First-class multi-room presidential suites yielded fares up to £512,
  whereas the upper IQR bound is ~£65.6. To keep these influential records without letting extreme skew distort linear estimators and variance,
  we apply soft upper-capping at the 99th percentile (~£249.0) rather than deleting rows.

In [17]:
# Helper function to detect outliers via IQR
def detect_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return lower_bound, upper_bound, len(outliers)


for col in ["Age", "Fare"]:
    lb, ub, count = detect_iqr_outliers(df[col])
    pct = (count / len(df)) * 100
    print(
        f"{col} Outlier Audit -> Lower: {lb:.2f}, Upper: {ub:.2f} | Outlier count: {count} ({pct:.2f}%)"
    )

# Cap extreme Fare outliers at the 99th percentile to retain data while reducing skew
fare_99th = df["Fare"].quantile(0.99)
print(f"\nCapping Fare at the 99th percentile (£{fare_99th:.2f})...")
df["Fare_Original"] = df["Fare"]  # Backup
df["Fare"] = df["Fare"].clip(upper=fare_99th)

print(f"Max fare after capping: £{df['Fare'].max():.2f}")

Age Outlier Audit -> Lower: 0.00, Upper: 56.00 | Outlier count: 35 (3.93%)
Fare Outlier Audit -> Lower: -26.72, Upper: 65.63 | Outlier count: 116 (13.02%)

Capping Fare at the 99th percentile (£249.01)...
Max fare after capping: £249.01


In [ ]:
## Step 6: Data Type Corrections

We enforce semantic data types across all columns:
- PassengerId $\rightarrow$ string (identifier, not arithmetic quantity)
- Survived $\rightarrow$ int8 / category (binary indicator)
- Pclass $\rightarrow$ category (ordered sociological class 1st, 2nd, 3rd)
- Age $\rightarrow$ float32 (rounded to 1 decimal place)
- SibSp, Parch $\rightarrow$ int8 (discrete counts)
- Fare $\rightarrow$ float64 (currency value rounded to 2 decimals)
- Sex, Embarked, Deck, Title $\rightarrow$ category

In [18]:
# Cast data types
df["PassengerId"] = df["PassengerId"].astype(str)
df["Survived"] = df["Survived"].astype("int8")
df["Pclass"] = pd.Categorical(
    df["Pclass"], categories=[1, 2, 3], ordered=True
)
df["Age"] = df["Age"].round(1).astype("float32")
df["SibSp"] = df["SibSp"].astype("int8")
df["Parch"] = df["Parch"].astype("int8")
df["Fare"] = df["Fare"].round(2).astype("float64")
df["Sex"] = df["Sex"].astype("category")
df["Embarked"] = df["Embarked"].astype("category")
df["Deck"] = df["Deck"].astype("category")
df["Title"] = df["Title"].astype("category")

print("Updated column datatypes:")
print(df.dtypes)

Updated column datatypes:
PassengerId           str
Survived             int8
Pclass           category
Name                  str
Sex              category
Age               float32
SibSp                int8
Parch                int8
Ticket                str
Fare              float64
Cabin                 str
Embarked         category
Title            category
Deck             category
Fare_Original     float64
dtype: object


In [ ]:
#3# Step 7: Before vs After Summary Comparison

To verify cleaning integrity, we compare the dataset state before and after our cleaning pipeline across null counts, duplicates,
row totals, and typing accuracy.

In [19]:
# Construct Before vs After summary comparison table
before_metrics = {
    "Total Rows": len(df_raw),
    "Total Columns": len(df_raw.columns),
    "Duplicate Rows": df_raw.duplicated().sum(),
    "Missing Age": df_raw["Age"].isnull().sum(),
    "Missing Cabin": df_raw["Cabin"].isnull().sum(),
    "Missing Embarked": df_raw["Embarked"].isnull().sum(),
    "Sex Dtype": str(df_raw["Sex"].dtype),
    "PassengerId Dtype": str(df_raw["PassengerId"].dtype),
    "Pclass Dtype": str(df_raw["Pclass"].dtype),
    "Max Fare": f"£{df_raw['Fare'].max():.2f}",
}

after_metrics = {
    "Total Rows": len(df),
    "Total Columns": len(df.columns),
    "Duplicate Rows": df.duplicated().sum(),
    "Missing Age": df["Age"].isnull().sum(),
    "Missing Cabin": (df["Cabin"] == "Unknown").sum(),  # cataloged explicitly
    "Missing Embarked": df["Embarked"].isnull().sum(),
    "Sex Dtype": str(df["Sex"].dtype),
    "PassengerId Dtype": str(df["PassengerId"].dtype),
    "Pclass Dtype": str(df["Pclass"].dtype),
    "Max Fare": f"£{df['Fare'].max():.2f}",
}

comparison_df = pd.DataFrame(
    [before_metrics, after_metrics],
    index=["Before Cleaning", "After Cleaning"],
).T

comparison_df

,Before Cleaning,After Cleaning
Total Rows,891,891
Total Columns,12,15
Duplicate Rows,0,0
Missing Age,177,0
Missing Cabin,687,687
Missing Embarked,2,0
Sex Dtype,str,category
PassengerId Dtype,int64,str
Pclass Dtype,int64,category
Max Fare,£512.33,£249.01


In [ ]:
### Step 8 Save Cleaned Dataset to CSV

Finally, we export the claned dataset into titanic_cleaned.csv ready for statistical modling or visualization pipelines.

In [20]:
# Export cleaned dataset
output_filename = "titanic_cleaned.csv"
df.to_csv(output_filename, index=False)
print(
    f"Successfully exported cleaned dataset to '{output_filename}' ({len(df)} rows, {len(df.columns)} columns)."
)

# Preview final 5 rows
df.head()

Successfully exported cleaned dataset to 'titanic_cleaned.csv' (891 rows, 15 columns).


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Deck,Fare_Original
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.25,Unknown,S,Mr,Unknown,7.2500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,71.28,C85,C,Mrs,C,71.2833
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.92,Unknown,S,Miss,Unknown,7.9250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.10,C123,S,Mrs,C,53.1000
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.05,Unknown,S,Mr,Unknown,8.0500
